# 🎬 Manim Video Production — Notebook

Notebook chính để tạo, preview và lưu video Manim.  
Chạy từng cell theo thứ tự, hoặc nhảy đến scene cần render.

---
## 1. Cài đặt & Khởi tạo
Chạy cell này **một lần** khi mở notebook.

In [ ]:
# Cài đặt thư viện (bỏ comment nếu chưa cài)
# !pip install manim numpy

In [ ]:
import sys
import os

# Đảm bảo project root nằm trong sys.path
PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import Manim
import manim
MANIM_VERSION = manim.__version__
from manim import *

# Import project modules
from config import *
from utils import *
from components import *

print(f"✅ Manim version: {MANIM_VERSION}")
print(f"✅ Project root: {PROJECT_ROOT}")
print(f"✅ All modules imported successfully!")

---
## 2. Cấu hình Render
Thay đổi chất lượng render tại đây.

In [ ]:
# ╔══════════════════════════════════════════════╗
# ║  CẤU HÌNH RENDER — Thay đổi tại đây         ║
# ╚══════════════════════════════════════════════╝

# Chất lượng render trong notebook:
#   "low"    → 480p, 15fps  (preview nhanh)
#   "medium" → 720p, 30fps  (xem thử)
#   "high"   → 1080p, 60fps (render final)
#   "fourk"  → 2160p, 60fps (4K)

RENDER_QUALITY = "low"  # ← Thay đổi tại đây

# Mapping quality → config
_quality_map = {
    "low":    {"pixel_height": 480,  "pixel_width": 854,  "frame_rate": 15},
    "medium": {"pixel_height": 720,  "pixel_width": 1280, "frame_rate": 30},
    "high":   {"pixel_height": 1080, "pixel_width": 1920, "frame_rate": 60},
    "fourk":  {"pixel_height": 2160, "pixel_width": 3840, "frame_rate": 60},
}

_q = _quality_map[RENDER_QUALITY]
config.pixel_height = _q["pixel_height"]
config.pixel_width = _q["pixel_width"]
config.frame_rate = _q["frame_rate"]

print(f"🎥 Render quality: {RENDER_QUALITY} ({_q['pixel_width']}x{_q['pixel_height']}, {_q['frame_rate']}fps)")

---
## 3. 🎬 Scene 1 — Homophily vs Heterophily
Part 1: Why GNN is strong/weak depending on graph structure

In [ ]:
%%manim -v WARNING Scene1_HomophilyHeterophily

class Scene1_HomophilyHeterophily(Scene):

    def construct(self):
        apply_scene_config(self)

        # ============================================================
        # PHASE 1: TITLE & FORMULA
        # ============================================================
        title = Text(
            "Local Homophily",
            font_size=36,
            color=TEXT_PRIMARY,
            weight=BOLD,
        )
        title.to_edge(UP, buff=0.5)

        formula = MathTex(
            r"h_v",
            r"=",
            r"\frac{1}{|\mathcal{N}(v)|}",
            r"\sum_{u \in \mathcal{N}(v)}",
            r"\mathbf{1}[y_u = y_v]",
            font_size=40,
        )
        formula.next_to(title, DOWN, buff=0.35)

        # --- Show Title ---
        self.play(fade_in_shift(title, direction=DOWN), run_time=0.8)
        self.wait(WAIT_SHORT)

        # --- Write Formula ---
        self.play(Write(formula), run_time=2.0)
        self.wait(WAIT_LONG)

        # --- Move & scale formula to top-left corner (clearly readable) ---
        title_formula = VGroup(title, formula)
        self.play(
            title_formula.animate.scale(0.72).to_corner(UL, buff=0.45),
            run_time=ANIM_SPEED_NORMAL,
        )
        self.wait(WAIT_SHORT)

        # ============================================================
        # PHASE 2: BUILD TWO GRAPHS (Left & Right)
        # ============================================================

        CENTER_R = 0.38          # Radius for center node
        NEIGHBOR_R = 0.28        # Radius for neighbor nodes
        ORBIT = 1.55             # Distance neighbor -> center
        GRAPH_Y = -0.35          # Vertical center of graph

        # ==========================
        # LEFT GRAPH: HOMOPHILY
        # ==========================
        left_pos = np.array([-3.3, GRAPH_Y, 0])

        l_center = Circle(
            radius=CENTER_R, color=GREEN,
            fill_color=GREEN_E, fill_opacity=0.9, stroke_width=3,
        ).move_to(left_pos)
        l_center_v = Text("v", font_size=20, color=WHITE, weight=BOLD)
        l_center_v.move_to(left_pos)

        # 4 neighbors (all GREEN) - 45°, 135°, 225°, 315°
        l_angles = [PI / 4, 3 * PI / 4, 5 * PI / 4, 7 * PI / 4]
        l_neighbors = []
        l_edges = []

        for angle in l_angles:
            pos = left_pos + np.array([
                np.cos(angle), np.sin(angle), 0
            ]) * ORBIT

            node = Circle(
                radius=NEIGHBOR_R, color=GREEN,
                fill_color=GREEN_D, fill_opacity=0.75, stroke_width=2.5,
            ).move_to(pos)
            l_neighbors.append(node)

            edge = Line(
                left_pos, pos,
                color=EDGE_COLOR, stroke_width=3,
            )
            l_edges.append(edge)

        l_label = Text(
            "Homophily: GNN Strong",
            font_size=22, color=GREEN, weight=BOLD,
        )
        l_all_nodes = VGroup(l_center, *l_neighbors)
        l_label.next_to(l_all_nodes, DOWN, buff=0.45)

        # ==========================
        # RIGHT GRAPH: HETEROPHILY
        # ==========================
        right_pos = np.array([3.3, GRAPH_Y, 0])

        r_center = Circle(
            radius=CENTER_R, color=GREEN,
            fill_color=GREEN_E, fill_opacity=0.9, stroke_width=3,
        ).move_to(right_pos)
        r_center_v = Text("v", font_size=20, color=WHITE, weight=BOLD)
        r_center_v.move_to(right_pos)

        # 3 neighbors: RED, YELLOW, PURPLE - 90°, 210°, 330°
        r_colors = [RED, YELLOW, PURPLE]
        r_fill_colors = [RED_E, YELLOW_E, PURPLE_E]
        r_angles = [PI / 2, 7 * PI / 6, 11 * PI / 6]
        r_neighbors = []
        r_edges = []

        for angle, nc, fc in zip(r_angles, r_colors, r_fill_colors):
            pos = right_pos + np.array([
                np.cos(angle), np.sin(angle), 0
            ]) * ORBIT

            node = Circle(
                radius=NEIGHBOR_R, color=nc,
                fill_color=fc, fill_opacity=0.75, stroke_width=2.5,
            ).move_to(pos)
            r_neighbors.append(node)

            edge = Line(
                right_pos, pos,
                color=EDGE_COLOR, stroke_width=3,
            )
            r_edges.append(edge)

        r_label = Text(
            "Heterophily: GNN Weak",
            font_size=22, color=RED, weight=BOLD,
        )
        r_all_nodes = VGroup(r_center, *r_neighbors)
        r_label.next_to(r_all_nodes, DOWN, buff=0.45)

        # --- Divider line between graphs ---
        divider = DashedLine(
            UP * 3, DOWN * 3.5,
            color=TEXT_MUTED, stroke_width=1.5, dash_length=0.15,
        )

        # ==========================
        # ANIMATE: Create Graphs
        # ==========================
        self.play(Create(divider), run_time=0.4)

        self.play(
            *[Create(e) for e in l_edges + r_edges],
            run_time=ANIM_SPEED_NORMAL,
        )

        all_nodes = [l_center] + l_neighbors + [r_center] + r_neighbors
        self.play(
            *[GrowFromCenter(n) for n in all_nodes],
            run_time=ANIM_SPEED_NORMAL,
        )

        self.play(
            FadeIn(l_center_v, scale=0.5),
            FadeIn(r_center_v, scale=0.5),
            run_time=0.4,
        )

        self.play(
            fade_in_shift(l_label, direction=UP, shift_distance=0.3),
            fade_in_shift(r_label, direction=UP, shift_distance=0.3),
        )
        self.wait(WAIT_LONG)

        # ============================================================
        # PHASE 3: ENHANCED DYNAMIC MOTION & MESSAGE PASSING
        # ============================================================

        # Step 1: Pulse / Highlight neighbor nodes (Feature readiness)
        self.play(
            *[
                Indicate(n, scale_factor=1.2, color=n.get_color())
                for n in l_neighbors + r_neighbors
            ],
            run_time=0.8,
        )
        self.wait(0.2)

        # Step 2: Multi-packet data stream moving along edges from neighbors to center
        DOT_R = 0.09

        # Wave 1: First batch of message particles
        wave1_l_dots = [
            Dot(radius=DOT_R, color=GREEN_B, fill_opacity=1).move_to(nb.get_center())
            for nb in l_neighbors
        ]
        wave1_r_dots = [
            Dot(radius=DOT_R, color=nc, fill_opacity=1).move_to(nb.get_center())
            for nb, nc in zip(r_neighbors, r_colors)
        ]

        # Wave 2: Second trailing batch for fluid streaming effect
        wave2_l_dots = [
            Dot(radius=DOT_R * 0.8, color=GREEN_A, fill_opacity=0.85).move_to(nb.get_center())
            for nb in l_neighbors
        ]
        wave2_r_dots = [
            Dot(radius=DOT_R * 0.8, color=nc, fill_opacity=0.85).move_to(nb.get_center())
            for nb, nc in zip(r_neighbors, r_colors)
        ]

        # Edge flashing effect to show active data transfer channels
        l_flash_edges = [
            Line(nb.get_center(), l_center.get_center(), color=GREEN_B, stroke_width=4.5)
            for nb in l_neighbors
        ]
        r_flash_edges = [
            Line(nb.get_center(), r_center.get_center(), color=nc, stroke_width=4.5)
            for nb, nc in zip(r_neighbors, r_colors)
        ]

        # Spawn wave 1 particles + edge glowing pulses
        self.play(
            *[FadeIn(d, scale=0.4) for d in wave1_l_dots + wave1_r_dots],
            *[ShowPassingFlash(e, time_width=0.4, run_time=1.4) for e in l_flash_edges + r_flash_edges],
            *[
                MoveAlongPath(
                    dot,
                    Line(dot.get_center(), l_center.get_center()),
                    rate_func=smooth,
                    run_time=1.4,
                )
                for dot in wave1_l_dots
            ],
            *[
                MoveAlongPath(
                    dot,
                    Line(dot.get_center(), r_center.get_center()),
                    rate_func=smooth,
                    run_time=1.4,
                )
                for dot in wave1_r_dots
            ],
        )

        # Wave 2 stream follows immediately
        self.play(
            *[FadeIn(d, scale=0.4) for d in wave2_l_dots + wave2_r_dots],
            *[
                MoveAlongPath(
                    dot,
                    Line(dot.get_center(), l_center.get_center()),
                    rate_func=smooth,
                    run_time=1.1,
                )
                for dot in wave2_l_dots
            ],
            *[
                MoveAlongPath(
                    dot,
                    Line(dot.get_center(), r_center.get_center()),
                    rate_func=smooth,
                    run_time=1.1,
                )
                for dot in wave2_r_dots
            ],
        )

        # Clean up stream dots
        self.remove(*wave1_l_dots, *wave1_r_dots, *wave2_l_dots, *wave2_r_dots)

        # Step 3: Center nodes reaction to aggregated messages
        # - Left (Homophily): Harmonic resonance, expanding green flash ring (Information aligned)
        # - Right (Heterophily): Conflicting signal collision, warning wobble & noise flash
        self.play(
            Circumscribe(
                l_center, color=GREEN_A, stroke_width=4,
                fade_out=True, run_time=1.5,
            ),
            Flash(l_center, color=GREEN_B, line_length=0.25, num_lines=10, run_time=1.2),
            l_center.animate.set_stroke(color=GREEN_A, width=4.5),
            Wiggle(
                r_center, scale_value=1.3,
                rotation_angle=0.08 * TAU, n_wiggles=6, run_time=1.5,
            ),
            Flash(r_center, color=RED_B, line_length=0.2, num_lines=8, run_time=1.2),
            r_center.animate.set_stroke(color=RED_B, width=4.5),
        )
        self.wait(WAIT_MEDIUM)

        # ============================================================
        # PHASE 4: SHOW h_v VALUES & BADGES
        # ============================================================
        hv_left = MathTex(r"h_v \to 1", font_size=32, color=GREEN)
        hv_left.next_to(l_label, DOWN, buff=0.3)

        hv_right = MathTex(r"h_v \to 0", font_size=32, color=RED)
        hv_right.next_to(r_label, DOWN, buff=0.3)

        # Subtle highlight box around result
        hv_left_box = SurroundingRectangle(
            hv_left, color=GREEN, buff=0.15, stroke_width=1.5, corner_radius=0.1
        )
        hv_right_box = SurroundingRectangle(
            hv_right, color=RED, buff=0.15, stroke_width=1.5, corner_radius=0.1
        )

        self.play(
            FadeIn(hv_left, shift=UP * 0.2),
            Create(hv_left_box),
            FadeIn(hv_right, shift=UP * 0.2),
            Create(hv_right_box),
            run_time=0.9,
        )
        self.wait(WAIT_EXTRA_LONG)

In [13]:
# 💾 LƯU VIDEO SCENE 1 VỪA CHẠY XONG VÀO THƯ MỤC EXPORTS/
import os, shutil, glob

def save_latest_video(filename="Scene1_HomophilyHeterophily.mp4", output_dir="exports"):
    if not filename.endswith(".mp4"):
        filename += ".mp4"
    os.makedirs(output_dir, exist_ok=True)
    mp4_files = [
        f for f in glob.glob("media/**/*.mp4", recursive=True)
        if "partial_movie_files" not in f and not os.path.basename(f).startswith(".")
    ]
    if not mp4_files:
        print("⚠️ Chưa tìm thấy file video nào trong media/. Hãy chạy render scene trước!")
        return None
    latest_file = max(mp4_files, key=os.path.getmtime)
    dest_path = os.path.join(output_dir, filename)
    shutil.copy2(latest_file, dest_path)
    file_size_mb = os.path.getsize(dest_path) / (1024 * 1024)
    abs_dest = os.path.abspath(dest_path)
    print("=" * 60)
    print(f"✅ ĐÃ LƯU VIDEO THÀNH CÔNG!")
    print(f"📁 Tên file  : {filename}")
    print(f"💾 Vị trí    : {abs_dest}")
    print(f"📦 Kích thước: {file_size_mb:.2f} MB")
    print("=" * 60)
    try:
        os.startfile(os.path.abspath(output_dir))
    except Exception:
        pass
    return abs_dest

# Thực thi lưu video và mở thư mục:
save_latest_video("Scene1_HomophilyHeterophily.mp4")

✅ ĐÃ LƯU VIDEO THÀNH CÔNG!
📁 Tên file  : Scene1_HomophilyHeterophily.mp4
💾 Vị trí    : c:\Users\ADMIN\Desktop\Video Manim\exports\Scene1_HomophilyHeterophily.mp4
📦 Kích thước: 1.03 MB


'c:\\Users\\ADMIN\\Desktop\\Video Manim\\exports\\Scene1_HomophilyHeterophily.mp4'

In [19]:
%%manim -v WARNING Scene1_ArchitectureOverview

class Scene1_ArchitectureOverview(Scene):
    """
    Sơ đồ hệ thống GLANCE (Figure 2 - Node-aware Fusion)
    - 3 Bước: Step 1 (Router) -> Step 2 (LLM Embedding) -> Step 3 (Refiner)
    - Frozen Backbones: GNN & LLM (Màu Cyan / Bông tuyết)
    - Trainable Modules: Router & Refiner (Vòng sáng Circumscribe + Màu Vàng)
    """

    def construct(self):
        apply_scene_config(self)

        # ============================================================
        # PHASE 1: TITLE & SUBTITLE
        # ============================================================
        title = Text(
            "GLANCE Architecture (Node-aware Fusion)",
            font_size=32,
            color=TEXT_PRIMARY,
            weight=BOLD,
        )
        title.to_edge(UP, buff=0.45)

        subtitle = Text(
            "Efficiency & Node-aware Adaptive Integration",
            font_size=15,
            color=TEXT_SECONDARY,
        )
        subtitle.next_to(title, DOWN, buff=0.12)

        title_group = VGroup(title, subtitle)
        self.play(FadeIn(title_group, shift=DOWN * 0.3), run_time=0.8)
        self.wait(WAIT_SHORT)

        # ============================================================
        # PHASE 2: 3 MAIN PIPELINE BOXES (Router, LLM, Refiner)
        # ============================================================
        BOX_W, BOX_H = 3.3, 2.05
        BOX_Y = -0.35

        # --- BOX 1: Step 1: Router (Green) ---
        b1_rect = RoundedRectangle(
            corner_radius=0.15, width=BOX_W, height=BOX_H,
            color=GREEN, fill_color="#0f2d1e", fill_opacity=0.9, stroke_width=2.5,
        )
        b1_badge = Text("STEP 1", font_size=11, color=GREEN_B, weight=BOLD)
        b1_title = Text("Router", font_size=22, color=TEXT_PRIMARY, weight=BOLD)
        b1_sub = Text("Feature Extraction\n& Node Routing", font_size=13, color=TEXT_SECONDARY, line_spacing=0.8)
        b1_content = VGroup(b1_badge, b1_title, b1_sub).arrange(DOWN, buff=0.12)
        b1 = VGroup(b1_rect, b1_content).move_to(LEFT * 4.3 + UP * BOX_Y)

        # --- BOX 2: Step 2: LLM Embedding (Blue) ---
        b2_rect = RoundedRectangle(
            corner_radius=0.15, width=BOX_W, height=BOX_H,
            color=BLUE, fill_color="#0d213a", fill_opacity=0.9, stroke_width=2.5,
        )
        b2_badge = Text("STEP 2", font_size=11, color=BLUE_B, weight=BOLD)
        b2_title = Text("LLM Embedding", font_size=20, color=TEXT_PRIMARY, weight=BOLD)
        b2_sub = Text("Multi-layer Layer\nRepresentations", font_size=13, color=TEXT_SECONDARY, line_spacing=0.8)
        b2_content = VGroup(b2_badge, b2_title, b2_sub).arrange(DOWN, buff=0.12)
        b2 = VGroup(b2_rect, b2_content).move_to(UP * BOX_Y)

        # --- BOX 3: Step 3: Refiner (Purple) ---
        b3_rect = RoundedRectangle(
            corner_radius=0.15, width=BOX_W, height=BOX_H,
            color=PURPLE, fill_color="#241238", fill_opacity=0.9, stroke_width=2.5,
        )
        b3_badge = Text("STEP 3", font_size=11, color=PURPLE_B, weight=BOLD)
        b3_title = Text("Refiner", font_size=22, color=TEXT_PRIMARY, weight=BOLD)
        b3_sub = Text("Prediction Fusion\n& Knowledge Mix", font_size=13, color=TEXT_SECONDARY, line_spacing=0.8)
        b3_content = VGroup(b3_badge, b3_title, b3_sub).arrange(DOWN, buff=0.12)
        b3 = VGroup(b3_rect, b3_content).move_to(RIGHT * 4.3 + UP * BOX_Y)

        # --- Connecting Arrows ---
        arrow1 = Arrow(
            b1_rect.get_right(), b2_rect.get_left(),
            buff=0.12, color="#64748b", stroke_width=3.5, max_tip_length_to_length_ratio=0.28,
        )
        arrow2 = Arrow(
            b2_rect.get_right(), b3_rect.get_left(),
            buff=0.12, color="#64748b", stroke_width=3.5, max_tip_length_to_length_ratio=0.28,
        )

        # Sequential animation
        self.play(Create(b1_rect), FadeIn(b1_content, shift=UP * 0.2), run_time=0.7)
        self.play(GrowArrow(arrow1), run_time=0.4)
        self.play(Create(b2_rect), FadeIn(b2_content, shift=UP * 0.2), run_time=0.7)
        self.play(GrowArrow(arrow2), run_time=0.4)
        self.play(Create(b3_rect), FadeIn(b3_content, shift=UP * 0.2), run_time=0.7)
        self.wait(WAIT_SHORT)

        # ============================================================
        # PHASE 3: FROZEN BACKBONES (GNN & LLM)
        # ============================================================
        frozen_llm_badge = VGroup(
            RoundedRectangle(
                corner_radius=0.1, width=2.4, height=0.55,
                color="#06b6d4", fill_color="#083344", fill_opacity=0.95, stroke_width=1.5,
            ),
            Text("❄ Frozen LLM", font_size=13, color="#22d3ee", weight=BOLD),
        )
        frozen_llm_badge[1].move_to(frozen_llm_badge[0].get_center())
        frozen_llm_badge.next_to(b2_rect, UP, buff=0.25)

        frozen_gnn_badge = VGroup(
            RoundedRectangle(
                corner_radius=0.1, width=2.4, height=0.55,
                color="#06b6d4", fill_color="#083344", fill_opacity=0.95, stroke_width=1.5,
            ),
            Text("❄ Frozen GNN", font_size=13, color="#22d3ee", weight=BOLD),
        )
        frozen_gnn_badge[1].move_to(frozen_gnn_badge[0].get_center())
        frozen_gnn_badge.move_to(frozen_llm_badge.get_center() + UP * 0.7)

        frozen_label = Text("Frozen Backbones (Zero Gradient Update)", font_size=14, color="#06b6d4", weight=SEMIBOLD)
        frozen_label.next_to(frozen_gnn_badge, UP, buff=0.18)

        self.play(
            FadeIn(frozen_gnn_badge, shift=DOWN * 0.3),
            FadeIn(frozen_llm_badge, shift=DOWN * 0.3),
            FadeIn(frozen_label),
            b2_rect.animate.set_stroke(color="#06b6d4", width=3.5),
            run_time=1.0,
        )
        self.wait(WAIT_MEDIUM)

        # ============================================================
        # PHASE 4: TRAINABLE MODULES HIGHLIGHT (Router & Refiner)
        # ============================================================
        trainable_label = Text("★ Trainable Modules (Lightweight ~ Few Parameters)", font_size=15, color=YELLOW, weight=BOLD)
        trainable_label.to_edge(DOWN, buff=0.5)

        trainable_box = SurroundingRectangle(
            trainable_label, color=YELLOW, buff=0.15, stroke_width=1.5, corner_radius=0.1, fill_color="#2d1f05", fill_opacity=0.65,
        )
        trainable_group = VGroup(trainable_box, trainable_label)

        # Glowing ring circumscribing Router & Refiner + Show Trainable label
        self.play(
            Circumscribe(b1_rect, color=YELLOW, stroke_width=4, time_width=0.6),
            Circumscribe(b3_rect, color=YELLOW, stroke_width=4, time_width=0.6),
            FadeIn(trainable_group, shift=UP * 0.2),
            b1_rect.animate.set_stroke(color=YELLOW, width=3.5),
            b3_rect.animate.set_stroke(color=YELLOW, width=3.5),
            run_time=1.4,
        )
        self.wait(WAIT_SHORT)

        # ============================================================
        # PHASE 5: END-TO-END DATA FLOW ANIMATION
        # ============================================================
        p1 = Dot(radius=0.09, color=YELLOW).move_to(b1_rect.get_left())
        path = Line(b1_rect.get_left(), b3_rect.get_right())

        self.play(
            MoveAlongPath(p1, path, rate_func=linear, run_time=1.8),
            Flash(b1_rect.get_center(), color=GREEN, line_length=0.25, run_time=0.6),
            Flash(b2_rect.get_center(), color=BLUE, line_length=0.25, run_time=0.6),
            Flash(b3_rect.get_center(), color=PURPLE, line_length=0.25, run_time=0.6),
        )
        self.remove(p1)
        self.wait(WAIT_LONG)


Manim Community v0.20.1

In [18]:
save_latest_video("Scene1_ArchitectureOverview.mp4")


✅ ĐÃ LƯU VIDEO THÀNH CÔNG!
📁 Tên file  : Scene1_ArchitectureOverview.mp4
💾 Vị trí    : c:\Users\ADMIN\Desktop\Video Manim\exports\Scene1_ArchitectureOverview.mp4
📦 Kích thước: 0.82 MB


'c:\\Users\\ADMIN\\Desktop\\Video Manim\\exports\\Scene1_ArchitectureOverview.mp4'

---
## 4. Scene mẫu — Demo Components
Chạy cell dưới để xem demo toàn bộ components có sẵn.

In [ ]:
%%manim -v WARNING ExampleScene

class ExampleScene(Scene):

    def construct(self):
        apply_scene_config(self)

        title = StyledTitle("Manim Boilerplate")
        subtitle = StyledSubtitle("Demo Components")
        title_group = VGroup(title, subtitle).arrange(DOWN, buff=0.4)

        self.play(fade_in_shift(title, direction=DOWN))
        self.play(fade_in_shift(subtitle, direction=UP))
        self.wait(WAIT_LONG)
        self.play(FadeOut(title_group))

        # --- Color Palette ---
        palette_title = StyledHeading("Color Palette")
        palette_title.to_edge(UP, buff=0.5)
        self.play(fade_in_shift(palette_title))

        color_squares = []
        colors_demo = [PRIMARY, SECONDARY, ACCENT, SUCCESS, WARNING, ERROR, INFO]
        names_demo = ["Primary", "Secondary", "Accent", "Success", "Warning", "Error", "Info"]

        for c, name in zip(colors_demo, names_demo):
            sq = Square(side_length=0.8, fill_color=c, fill_opacity=0.9, stroke_width=0)
            label = Text(name, font_size=14, color=TEXT_PRIMARY)
            label.next_to(sq, DOWN, buff=0.15)
            color_squares.append(VGroup(sq, label))
        palette = arrange_row(color_squares, buff=0.4)
        palette.next_to(palette_title, DOWN, buff=0.8)

        self.play(sequential_fade_in(color_squares, lag_ratio=0.1))
        self.wait(WAIT_LONG)
        self.play(*[FadeOut(m) for m in self.mobjects])

        outro = StyledTitle("Ready to Create!", color=PRIMARY_LIGHT)
        self.play(scale_fade_in(outro, start_scale=0.7))
        self.wait(WAIT_LONG)
        self.play(FadeOut(outro))

---
## 5. 📝 Template — Create New Scene

Copy the cell below, rename the class, and write your animation code.  
Each scene = one cell. Run the cell to render.

In [ ]:
%%manim -v WARNING MyScene

class MyScene(Scene):
    def construct(self):
        apply_scene_config(self)

        # ===== WRITE ANIMATION CODE HERE =====

        title = StyledTitle("Hello World")
        self.play(fade_in_shift(title))
        self.wait(2)
        self.play(FadeOut(title))

---
## 6. 🧰 Quick Reference

### Text Components
```python
StyledTitle("Big Title")           # Large bold title
StyledSubtitle("Subtitle")         # Subtitle
StyledHeading("Heading")           # Section heading
StyledBody("Body text")            # Body text
StyledCaption("Caption")           # Small caption
BulletList(["A", "B", "C"])        # Bullet list
SectionTitle("Title", "subtitle")  # Heading + underline
```

### Box Components
```python
InfoBox("Title", "Content text")   # Info box
HighlightBox(some_mob, color=ACCENT) # Surround object
CodeBlock("x = 42", language_label="Python") # Code block
GlassPanel(width=5, height=3)      # Glass panel
```

### Annotations
```python
LabeledArrow("label", start=LEFT, end=RIGHT)
BraceAnnotation(mob, "text", direction=DOWN)
Callout("Note!", target_mob, direction=UP)
DashedConnection(mob_a, mob_b)
```

### Custom Animations
```python
fade_in_shift(mob, direction=UP)
fade_out_shift(mob, direction=DOWN)
sequential_fade_in(mobs_list)
highlight_pulse(mob, color=ACCENT)
sweep_in(mob, direction=LEFT)
scale_fade_in(mob, start_scale=0.5)
typewriter_text(self, text_mob)
blink(mob, times=2)
draw_then_fade(mob)
```

### Layout & Export
```python
arrange_row([a, b, c], buff=0.5)
arrange_column([a, b, c], buff=0.5)
arrange_in_grid(items, rows=2, cols=3)
pin_to_edge(mob, UP, margin=0.5)
split_screen_left_right(left, right)
save_latest_video("my_video.mp4", output_dir="exports") # Lưu video
```

### Colors
```python
PRIMARY, SECONDARY, ACCENT
SUCCESS, WARNING, ERROR, INFO
TEXT_PRIMARY, TEXT_SECONDARY
BG_DARK, BG_MEDIUM, BG_LIGHT
NODE_COLORS[i] / get_node_color(index)
gradient_colors(start, end, steps)
```

---
## 7. 🎬 Export High Quality Video (Full HD 1080p / 4K)

In [ ]:
# Render chất lượng cao qua CLI
SCENE_FILE = "scenes/scene1_homophily_heterophily.py"
SCENE_CLASS = "Scene1_HomophilyHeterophily"
QUALITY = "-qh"  # -ql (480p) / -qm (720p) / -qh (1080p) / -qk (4K)

!manim {QUALITY} {SCENE_FILE} {SCENE_CLASS}

In [ ]:
# 💾 LƯU & MỞ VIDEO VỪA EXPORT CHẤT LƯỢNG CAO
save_latest_video("Scene1_FullHD_1080p.mp4", output_dir="exports")